# Notebook 04 — IT Feedback and Multi-Glance Loop (D1–D4)

**Ablations:** D1 (single glance) · D2 (multi-glance, soft/differentiable) ·
D3 (multi-glance, REINFORCE hard attention) · D4 (entropy halting + counterfactual)  
**New module:** `src/it_feedback.py` — `FoveatedModel`, `FixationGrid`, halters  
**Outputs:** `results/04_metrics.json` · `results/04_efficiency_curve.png` ·
`results/04_glance_trajectory.png`

---

## Purpose

The IT-feedback loop (component C5) enables **active, task-driven vision**: instead of
processing the full image uniformly, the model sequentially fixates high-information
locations, accumulates evidence, and halts when confidence is sufficient.

Three variants:
- **D1 (single glance):** standard feedforward, foveation at image centre only.
- **D2 (soft/differentiable):** multi-glance over a fixed grid, soft confidence
  weighting — fully differentiable, trained end-to-end.
- **D3 (REINFORCE):** fixation locations sampled from a learned policy; reward = correct
  classification. Gradient estimated via REINFORCE.
- **D4 (entropy + counterfactual):** uncertainty-driven halting +
  counterfactual loss that rewards informative fixations.

## Mathematics

**Confidence and entropy:**

$$
p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}, \quad
c = \max_i p_i, \quad
H(p) = -\sum_i p_i \log p_i
$$

**Halting condition (D2):** stop after glance $t$ if $c \ge \tau$ or $t = T_{\max}$.

**Soft evidence accumulation (D2):** differentiable weighted sum of glance logits:

$$
\bar z = \frac{1}{T}\sum_{t=1}^T z_t, \quad
\mathcal L = \mathrm{CE}(\bar z,\, y)
$$

**REINFORCE gradient (D3):**

$$
J = \mathbb{E}_{\pi_\phi}[R], \quad
\nabla_\phi J = \mathbb{E}\!\left[(R - b)\,\nabla_\phi \log \pi_\phi(l \mid s)\right]
$$

where $b$ is a baseline (running mean reward) to reduce variance.

**Counterfactual reward (D4):**

$$
r_t^{\mathrm{CF}} = p_{y}^{(t)} - p_{y}^{(t-1)} \quad \text{(correct-class prob gain per glance)}
$$


In [ ]:
# ── Environment ─────────────────────────────────────────────────────────────
import sys, os, warnings, json, importlib
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/tez_foveated_vision"
else:
    _here = os.getcwd()
    for _d in [_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))]:
        if os.path.isdir(os.path.join(_d, "src")):
            PROJECT_ROOT = _d; break
    else:
        PROJECT_ROOT = _here

os.chdir(PROJECT_ROOT)
for _p in [PROJECT_ROOT,
           os.path.join(PROJECT_ROOT, "external", "vonenet"),
           os.path.join(PROJECT_ROOT, "external", "CORnet")]:
    if _p not in sys.path: sys.path.insert(0, _p)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src import common, datasets, models, foveation as fov_mod
from src.foveation import apply_rblur, eccentricity_map, FoveatedTransform, build_foveated_transform, SNRProfile

CFG = {
    "seed": 42, "smoke_test": True,
    "dataset": "cifar10", "image_size": 32, "n_classes": 10,
    "ppd": 4.0, "fixation_yx": (0.5, 0.5),
    "fovea_deg": 1.0, "transition_deg": 0.5,
    "rblur_sigma0": 0.5, "rblur_slope": 1.5,
    "snr0_db": 30.0, "beta": 2.0, "patch_size": 8,
    # Multi-glance
    "max_glances": 4, "halt_threshold": 0.75,
    "fixation_grid_rows": 2, "fixation_grid_cols": 2,
    "margin": 0.25,
    # Training
    "n_epochs_smoke": 3, "n_batches_smoke": 20,
    "n_epochs_full": 20, "n_batches_full": None,
    "batch_size": 32, "lr": 1e-3,
    # CIFAR-10 mandatory (P1); ImageNet-100 for FoveaTer-style efficiency
    # comparison (docs/experiment-plan-and-ablations.md §4, Phase 4). ImageNet-100
    # is license-gated / not auto-downloaded -- honest fallback in build_real_loaders.
    "eval_datasets": ["cifar10", "imagenet100"],
    "result_dir": os.path.join(PROJECT_ROOT, "results"),
    "ckpt_dir":   os.path.join(PROJECT_ROOT, "checkpoints"),
    "data_dir":   os.path.join(PROJECT_ROOT, "data"),
}

common.set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CFG["result_dir"], exist_ok=True)
print(f"Device: {device} | Smoke test: {CFG['smoke_test']}")


## Implementation: FixationGrid and Multi-Glance Components

### Fixation grid

The simplest fixation policy (for D2) uses a regular $n_r \times n_c$ grid:

$$
l_{ij} = \left(\mathrm{margin} + \frac{i}{n_r - 1}(1 - 2\,\mathrm{margin}),\;
                \mathrm{margin} + \frac{j}{n_c - 1}(1 - 2\,\mathrm{margin})\right)
$$

Always includes the image centre $(0.5, 0.5)$ as the first fixation (D1 behaviour).

### Soft evidence accumulation

At each glance $t$ with fixation $l_t$:
1. Foveate image at $l_t$: $x_t = \mathrm{RBlur}(x,\, l_t)$
2. Classify: $z_t = f_\theta(x_t) \in \mathbb{R}^K$
3. Accumulate: $\bar z_t = \frac{1}{t}\sum_{s=1}^{t} z_s$
4. Halt if $\max_i \mathrm{softmax}(\bar z_t)_i \ge \tau$

Loss: $\mathcal{L} = \mathrm{CE}(\bar z_{T_{\mathrm{stop}}},\, y)$


In [ ]:
# ── Fixation grid and multi-glance foveated model ────────────────────────

class FixationGrid:
    """Regular fixation grid always including the centre as first fixation."""

    def __init__(self, n_rows=2, n_cols=2, margin=0.25):
        self.n_rows  = n_rows
        self.n_cols  = n_cols
        self.margin  = margin

    def get_fixations(self):
        """Return list of (fy, fx) fixation coords in [0,1]^2, centre first."""
        m = self.margin
        ys = [m + i / max(self.n_rows - 1, 1) * (1 - 2 * m)
              for i in range(self.n_rows)]
        xs = [m + j / max(self.n_cols - 1, 1) * (1 - 2 * m)
              for j in range(self.n_cols)]
        grid = [(y, x) for y in ys for x in xs]
        # Ensure (0.5, 0.5) is first (centre = D1 baseline)
        centre = (0.5, 0.5)
        grid = sorted(grid, key=lambda p: abs(p[0]-0.5)+abs(p[1]-0.5))
        return grid[:self.n_rows * self.n_cols]


class MultiGlanceFoveatedModel(nn.Module):
    """Differentiable multi-glance foveated classifier (Ablation D2).

    At each glance t:
      1. Foveate image at fixation l_t using R-Blur.
      2. Classify with a shared backbone f_theta.
      3. Accumulate mean logits: z_bar_t = mean(z_1...z_t).
      4. Halt if confidence max softmax(z_bar_t) >= threshold (eval only).

    Loss = CE(z_bar_T, y) -- fully differentiable w.r.t. backbone params.
    """

    def __init__(self, backbone_fn, fixations, halt_threshold=0.75,
                 sigma0=0.5, slope=1.5, ppd=4.0):
        super().__init__()
        self.backbone = backbone_fn          # nn.Module, raw [0,1] input
        self.fixations = fixations           # list of (fy, fx)
        self.halt_threshold = halt_threshold
        self.sigma0 = sigma0
        self.slope  = slope
        self.ppd    = ppd

    def foveate(self, x, fixation_yx):
        """Apply R-Blur at the given fixation to each image in the batch."""
        out = []
        for i in range(x.size(0)):
            out.append(apply_rblur(x[i], fixation_yx, self.sigma0,
                                   self.slope, self.ppd))
        return torch.stack(out)

    def forward(self, x_raw, return_glance_count=False):
        """
        x_raw: [B, C, H, W] raw [0,1] float tensor.
        Returns logits [B, K] and optionally glance_count [B].
        """
        B = x_raw.size(0)
        logit_sum = None
        glance_count = torch.ones(B, dtype=torch.long, device=x_raw.device)
        halted = torch.zeros(B, dtype=torch.bool, device=x_raw.device)

        for t, fix in enumerate(self.fixations):
            fov_x = self.foveate(x_raw, fix)              # [B, C, H, W]
            logits_t = self.backbone(fov_x)                # [B, K]

            if logit_sum is None:
                logit_sum = logits_t
            else:
                logit_sum = logit_sum + logits_t

            avg_logits = logit_sum / (t + 1)

            # Halting (eval mode only — training runs all glances)
            if not self.training and t < len(self.fixations) - 1:
                conf = F.softmax(avg_logits, dim=1).max(dim=1).values  # [B]
                newly_halted = (conf >= self.halt_threshold) & (~halted)
                glance_count += (~halted & ~newly_halted).long()
                halted |= newly_halted
                if halted.all():
                    break

        final_logits = logit_sum / len(self.fixations)
        if return_glance_count:
            return final_logits, glance_count
        return final_logits


# ── Quick test ───────────────────────────────────────────────────────────────
common.set_seed(CFG["seed"])
grid = FixationGrid(CFG["fixation_grid_rows"], CFG["fixation_grid_cols"], CFG["margin"])
fixations = grid.get_fixations()
print(f"Fixation grid ({CFG['fixation_grid_rows']}x{CFG['fixation_grid_cols']}): {fixations}")

S = CFG["image_size"]
dummy = torch.rand(2, 3, S, S)

# Verify foveation at different fixation points produces different images
for fix in fixations[:2]:
    fov_img = apply_rblur(dummy[0], fix, CFG["rblur_sigma0"], CFG["rblur_slope"], CFG["ppd"])
    print(f"  fix={fix}: foveated image norm = {fov_img.norm():.3f}")

print("✓ FixationGrid and foveation OK")

## REINFORCE Hard Attention (D3 — overview)

D3 uses a **learned fixation policy** $\pi_\phi(l \mid s)$ that samples the next
fixation location from a categorical distribution over grid cells:

$$
l_t \sim \pi_\phi(\cdot \mid s_t), \quad
s_t = f_{\mathrm{enc}}(\bar z_{t-1})
$$

The gradient estimator (REINFORCE with baseline):

$$
\nabla_\phi J = \frac{1}{N}\sum_{n=1}^N \sum_{t=1}^{T_n}
\bigl(R_n - b\bigr)\,\nabla_\phi \log \pi_\phi(l_t^{(n)} \mid s_t^{(n)})
$$

where $R_n \in \{0,1\}$ is the classification reward for episode $n$, and
$b = \mathbb{E}[R]$ is a running-mean baseline.

> **Smoke-test scope:** D3 training requires many episodes to converge and is
> excluded from the smoke-test loop. The class below shows the mechanism.


In [ ]:
# ── REINFORCE policy (D3): learned fixation-selection mechanism ──────────
# Real training loop (real data, real epochs) is in the cell after D1/D2 training,
# reusing D1's already-trained backbone as a frozen feature extractor -- see there
# for why (reinforce_step below only back-props through the policy, not the
# backbone, so the backbone needs to already be a competent classifier).

class FixationPolicy(nn.Module):
    """Lightweight policy network mapping state -> fixation distribution.

    state: accumulated mean logits z_bar (K-dim)
    output: categorical distribution over N_fix fixation locations
    """
    def __init__(self, logit_dim, n_fixations):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(logit_dim, 32), nn.ReLU(),
            nn.Linear(32, n_fixations),
        )
        self.baseline = 0.0    # running-mean reward baseline

    def forward(self, state):
        return F.softmax(self.net(state), dim=-1)   # [B, N_fix]


def reinforce_step(backbone, policy, fixations, x_raw, y,
                    normalise_fn, device_, sigma0=0.5, slope=1.5, ppd=4.0,
                    greedy=False):
    """One REINFORCE episode: sample (or, if greedy, argmax) a second fixation
    from the policy given first-glance evidence, compute reward + policy gradient.
    `greedy=True` is for evaluation (deterministic fixation choice, no exploration)."""
    B = x_raw.size(0)
    n_fix = len(fixations)

    # Initialise with centre fixation (D1-like first glance)
    fov0 = torch.stack([apply_rblur(x_raw[i], fixations[0], sigma0, slope, ppd)
                         for i in range(B)])
    x_norm0 = normalise_fn(fov0)
    with torch.no_grad():
        z_bar = backbone(x_norm0)

    # Choose second fixation from policy
    state = z_bar.detach()
    probs = policy(state)                          # [B, n_fix]
    dist  = torch.distributions.Categorical(probs)
    idx   = probs.argmax(dim=-1) if greedy else dist.sample()   # [B]
    log_p = dist.log_prob(idx)                     # [B]

    # Apply chosen fixation and classify
    fov_list = []
    for i in range(B):
        fix = fixations[idx[i].item()]
        fov_list.append(apply_rblur(x_raw[i], fix, sigma0, slope, ppd))
    fov_batch = torch.stack(fov_list)
    x_norm1 = normalise_fn(fov_batch)
    with torch.no_grad():
        z2 = backbone(x_norm1)
    logits = (z_bar + z2) / 2
    pred   = logits.argmax(dim=1)
    reward = (pred == y).float()                   # [B]  in {0, 1}

    if greedy:
        return None, reward.mean().item(), logits

    # REINFORCE gradient
    baseline = policy.baseline
    loss_policy = -((reward - baseline) * log_p).mean()

    # Update baseline (running mean)
    policy.baseline = 0.9 * baseline + 0.1 * reward.mean().item()

    return loss_policy, reward.mean().item(), logits

In [ ]:
# ── Real data loaders: CIFAR-10 (mandatory) + ImageNet-100 (efficiency comparison) ──

class TinyCNN(nn.Module):
    """AdaptiveAvgPool2d(4) makes this resolution-agnostic: works unchanged on
    CIFAR-10's 32x32 and ImageNet-100's 224x224 without architecture changes."""
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(4),
            nn.Flatten(), nn.Linear(64*4*4, n_classes),
        )
    def forward(self, x): return self.net(x)


INV_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
INV_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def denorm(x): return (x * INV_STD + INV_MEAN).clamp(0.0, 1.0)
def renorm(x): return (x - INV_MEAN) / INV_STD

# Per-dataset foveation geometry (ImageNet-100 follows VOneNet's ppd=28 @ 224px
# convention, matching the analogous override in 02_foveation_rblur_and_periphery.ipynb).
DATASET_GEOMETRY = {
    "cifar10":     dict(image_size=32,  ppd=4.0,  fovea_deg=1.0, rblur_slope=1.5, n_classes=10),
    "imagenet100": dict(image_size=224, ppd=28.0, fovea_deg=1.5, rblur_slope=2.0, n_classes=100),
}


def build_real_loaders(dataset_name, batch_size=32):
    """Honest-fallback real loaders (src/datasets.py). CIFAR-10 auto-downloads
    (~170 MB) when smoke_test=False. ImageNet-100 is license-gated -- must already
    be placed under data/imagenet100/{train,val}/<class>/*.JPEG; raises
    FileNotFoundError with the expected layout if missing and smoke_test=False."""
    geom = DATASET_GEOMETRY[dataset_name]
    if dataset_name == "cifar10":
        tr = datasets.get_cifar10(CFG["data_dir"], train=True, normalization="imagenet",
                                   download=True, smoke_test=CFG["smoke_test"])
        va = datasets.get_cifar10(CFG["data_dir"], train=False, normalization="imagenet",
                                   download=True, smoke_test=CFG["smoke_test"])
    elif dataset_name == "imagenet100":
        tr = datasets.get_imagenet100(CFG["data_dir"], split="train", normalization="imagenet",
                                       image_size=geom["image_size"], smoke_test=CFG["smoke_test"])
        va = datasets.get_imagenet100(CFG["data_dir"], split="val", normalization="imagenet",
                                       image_size=geom["image_size"], smoke_test=CFG["smoke_test"])
    else:
        raise ValueError(f"Unknown dataset_name: {dataset_name!r}")

    num_workers = 2 if not CFG["smoke_test"] else 0
    train_ld = torch.utils.data.DataLoader(tr, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_ld   = torch.utils.data.DataLoader(va, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_ld, val_ld

print("Dataset helpers ready (cifar10, imagenet100).")

In [ ]:
# ── D1 (single glance) vs D2 (multi-glance) training ─────────────────────

def train_epoch(model, loader, opt, device_, geom, n_batches=None,
                multi_glance=False):
    model.train()
    mu  = torch.tensor([0.485, 0.456, 0.406], device=device_).view(1,3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device_).view(1,3,1,1)
    correct = total = tot_loss = 0
    for bi, (xb, yb) in enumerate(loader):
        if n_batches and bi >= n_batches: break
        xb, yb = xb.to(device_), yb.to(device_)
        x_raw  = (xb * std + mu).clamp(0.0, 1.0)   # undo normalisation -> [0,1]

        if multi_glance:
            logits = model(x_raw)
        else:
            # D1: centre fixation only, re-normalise before backbone
            fov_x  = torch.stack([apply_rblur(x_raw[i], (0.5, 0.5),
                                              CFG["rblur_sigma0"], geom["rblur_slope"],
                                              geom["ppd"])
                                   for i in range(x_raw.size(0))])
            fov_norm = (fov_x - mu) / std
            logits = model(fov_norm)

        loss = F.cross_entropy(logits, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        tot_loss += loss.item() * xb.size(0)
        correct  += (logits.argmax(1) == yb).sum().item()
        total    += xb.size(0)
    return tot_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def eval_epoch(model, loader, device_, geom, multi_glance=False):
    model.eval()
    mu  = torch.tensor([0.485, 0.456, 0.406], device=device_).view(1,3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device_).view(1,3,1,1)
    correct = total = total_glances = 0
    for xb, yb in loader:
        xb, yb = xb.to(device_), yb.to(device_)
        x_raw = (xb * std + mu).clamp(0.0, 1.0)
        if multi_glance:
            logits, gc = model(x_raw, return_glance_count=True)
            total_glances += gc.sum().item()
        else:
            fov_x   = torch.stack([apply_rblur(x_raw[i], (0.5, 0.5),
                                               CFG["rblur_sigma0"], geom["rblur_slope"],
                                               geom["ppd"])
                                    for i in range(x_raw.size(0))])
            fov_norm = (fov_x - mu) / std
            logits   = model(fov_norm)
            total_glances += xb.size(0)   # always 1 glance
        correct += (logits.argmax(1) == yb).sum().item()
        total   += xb.size(0)
    avg_glances = total_glances / max(total, 1)
    return correct / max(total, 1), avg_glances


class NormBackbone(nn.Module):
    """Wraps a raw-[0,1]-input backbone to apply ImageNet normalisation internally."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.register_buffer("mu",  torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1))
    def forward(self, x_raw):
        return self.backbone((x_raw - self.mu) / self.std)


# ── Train D1 and D2 for each dataset in CFG["eval_datasets"] ─────────────
# CIFAR-10 is the mandatory P1 fast-iteration pass; ImageNet-100 (if present under
# data/imagenet100/) is the FoveaTer-style efficiency comparison for this notebook's
# Phase 4 (docs/experiment-plan-and-ablations.md §4).

n_epochs  = CFG["n_epochs_smoke"]  if CFG["smoke_test"] else CFG["n_epochs_full"]
n_batches = CFG["n_batches_smoke"] if CFG["smoke_test"] else CFG["n_batches_full"]

hist_d1_by_dataset = {}
hist_d2_by_dataset = {}
loaders_by_dataset  = {}   # kept for the glance-trajectory figure below
backbones_d1_by_dataset = {}   # kept as the frozen feature extractor for D3 (REINFORCE)

for dataset_name in CFG["eval_datasets"]:
    geom = DATASET_GEOMETRY[dataset_name]
    print(f"\n{'='*70}\nDataset: {dataset_name}  (n_classes={geom['n_classes']})\n{'='*70}")
    try:
        train_ld, val_ld = build_real_loaders(dataset_name, CFG["batch_size"])
    except FileNotFoundError as e:
        print(f"  Skipping {dataset_name}: {e}")
        continue
    loaders_by_dataset[dataset_name] = (train_ld, val_ld)

    common.set_seed(CFG["seed"])
    print(f"=== [{dataset_name}] D1: Single-glance foveated classifier ===")
    backbone_d1 = TinyCNN(geom["n_classes"]).to(device)
    opt_d1 = torch.optim.Adam(backbone_d1.parameters(), lr=CFG["lr"])
    hist_d1 = []
    for epoch in range(n_epochs):
        tr_loss, tr_acc = train_epoch(backbone_d1, train_ld, opt_d1, device, geom,
                                       n_batches=n_batches)
        val_acc, avg_gc = eval_epoch(backbone_d1, val_ld, device, geom, multi_glance=False)
        hist_d1.append({"epoch": epoch, "val_acc": val_acc, "avg_glances": avg_gc})
        print(f"  E{epoch+1}: loss={tr_loss:.4f} tr={tr_acc:.3f} val={val_acc:.3f} glances={avg_gc:.1f}")
    hist_d1_by_dataset[dataset_name] = hist_d1
    backbones_d1_by_dataset[dataset_name] = backbone_d1

    common.set_seed(CFG["seed"])
    print(f"\n=== [{dataset_name}] D2: Multi-glance foveated classifier ===")
    backbone_d2 = TinyCNN(geom["n_classes"]).to(device)

    # MultiGlanceFoveatedModel wraps backbone to accept raw [0,1] input
    mg_model = MultiGlanceFoveatedModel(
        backbone_fn=NormBackbone(backbone_d2),
        fixations=fixations,
        halt_threshold=CFG["halt_threshold"],
        sigma0=CFG["rblur_sigma0"],
        slope=geom["rblur_slope"],
        ppd=geom["ppd"],
    ).to(device)

    opt_d2 = torch.optim.Adam(backbone_d2.parameters(), lr=CFG["lr"])
    hist_d2 = []
    for epoch in range(n_epochs):
        tr_loss, tr_acc = train_epoch(mg_model, train_ld, opt_d2, device, geom,
                                       n_batches=n_batches, multi_glance=True)
        val_acc, avg_gc = eval_epoch(mg_model, val_ld, device, geom, multi_glance=True)
        hist_d2.append({"epoch": epoch, "val_acc": val_acc, "avg_glances": avg_gc})
        print(f"  E{epoch+1}: loss={tr_loss:.4f} tr={tr_acc:.3f} val={val_acc:.3f} glances={avg_gc:.2f}")
    hist_d2_by_dataset[dataset_name] = hist_d2

In [ ]:
# ── D3: REINFORCE fixation policy — real training ─────────────────────────
# Trains a learned fixation-selection policy on top of D1's already-trained
# (frozen) backbone, for each dataset in CFG["eval_datasets"]. The policy chooses
# a second glance location conditioned on the first glance's evidence (z_bar);
# reward = whether averaging both glances' logits gives the correct prediction.

n_epochs_d3  = CFG["n_epochs_smoke"]  if CFG["smoke_test"] else CFG["n_epochs_full"]
n_batches_d3 = CFG["n_batches_smoke"] if CFG["smoke_test"] else CFG["n_batches_full"]

hist_d3_by_dataset   = {}
policy_by_dataset    = {}

for dataset_name in CFG["eval_datasets"]:
    if dataset_name not in loaders_by_dataset:
        continue  # skipped earlier (e.g. ImageNet-100 not present locally)
    geom = DATASET_GEOMETRY[dataset_name]
    train_ld, val_ld = loaders_by_dataset[dataset_name]
    backbone_frozen = backbones_d1_by_dataset[dataset_name].eval()
    for p in backbone_frozen.parameters():
        p.requires_grad_(False)

    print(f"\n=== [{dataset_name}] D3: REINFORCE fixation policy ===")
    common.set_seed(CFG["seed"])
    policy_net = FixationPolicy(logit_dim=geom["n_classes"], n_fixations=len(fixations)).to(device)
    policy_opt = torch.optim.Adam(policy_net.parameters(), lr=1e-3)

    mu  = torch.tensor([0.485, 0.456, 0.406], device=device).view(1,3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1,3,1,1)
    normalise_fn = lambda x: (x - mu) / std

    n_batches_val_d3 = 10 if CFG["smoke_test"] else None
    history = []
    for epoch in range(n_epochs_d3):
        policy_net.train()
        ep_reward = ep_loss = n_seen = 0
        for bi, (xb, yb) in enumerate(train_ld):
            if n_batches_d3 and bi >= n_batches_d3: break
            xb, yb = xb.to(device), yb.to(device)
            x_raw = (xb * std + mu).clamp(0.0, 1.0)
            loss_p, reward, _ = reinforce_step(
                backbone_frozen, policy_net, fixations, x_raw, yb, normalise_fn, device,
                sigma0=CFG["rblur_sigma0"], slope=geom["rblur_slope"], ppd=geom["ppd"])
            policy_opt.zero_grad(); loss_p.backward(); policy_opt.step()
            ep_loss   += loss_p.item() * xb.size(0)
            ep_reward += reward * xb.size(0)
            n_seen    += xb.size(0)

        # Evaluate: greedy (argmax) fixation choice, no exploration
        policy_net.eval()
        val_correct = val_total = 0
        for bi, (xb, yb) in enumerate(val_ld):
            if n_batches_val_d3 and bi >= n_batches_val_d3: break
            xb, yb = xb.to(device), yb.to(device)
            x_raw = (xb * std + mu).clamp(0.0, 1.0)
            with torch.no_grad():
                _, _, logits = reinforce_step(
                    backbone_frozen, policy_net, fixations, x_raw, yb, normalise_fn, device,
                    sigma0=CFG["rblur_sigma0"], slope=geom["rblur_slope"], ppd=geom["ppd"],
                    greedy=True)
            val_correct += (logits.argmax(1) == yb).sum().item()
            val_total   += xb.size(0)
        val_acc = val_correct / max(val_total, 1)
        train_reward = ep_reward / max(n_seen, 1)
        history.append({"epoch": epoch, "train_reward": train_reward, "val_acc": val_acc,
                         "avg_glances": 2.0})   # centre glance + 1 policy-chosen glance, always
        print(f"  E{epoch+1}/{n_epochs_d3}: policy_loss={ep_loss/max(n_seen,1):.4f} "
              f"train_reward={train_reward:.3f} val_acc={val_acc:.3f}")

    hist_d3_by_dataset[dataset_name] = history
    policy_by_dataset[dataset_name] = policy_net

print("\n(D3 uses D1's frozen backbone as a fixed feature extractor; only the "
      "REINFORCE fixation policy is trained here.)")

In [ ]:
# ── Visualise: efficiency curve + glance trajectories (per dataset) ───────

common.set_seed(CFG["seed"])

for dataset_name in CFG["eval_datasets"]:
    if dataset_name not in hist_d1_by_dataset:
        continue  # skipped above (e.g. ImageNet-100 not present locally)
    hist_d1 = hist_d1_by_dataset[dataset_name]
    hist_d2 = hist_d2_by_dataset[dataset_name]
    hist_d3 = hist_d3_by_dataset.get(dataset_name)
    train_ld, val_ld = loaders_by_dataset[dataset_name]
    geom = DATASET_GEOMETRY[dataset_name]

    # Figure 1: accuracy vs avg glances
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    ax = axes[0]
    epochs = [h["epoch"] for h in hist_d1]
    ax.plot(epochs, [h["val_acc"] for h in hist_d1], "o-", label="D1 single-glance")
    ax.plot(epochs, [h["val_acc"] for h in hist_d2], "s-", label="D2 multi-glance")
    if hist_d3:
        ax.plot([h["epoch"] for h in hist_d3], [h["val_acc"] for h in hist_d3],
                 "^-", label="D3 REINFORCE")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Val accuracy")
    ax.set_title(f"D1 vs D2 vs D3: accuracy per epoch [{dataset_name}]"
                 + ("\n(smoke_test, random model)" if CFG["smoke_test"] else ""))
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    d1_gc = [h["avg_glances"] for h in hist_d1]
    d2_gc = [h["avg_glances"] for h in hist_d2]
    d1_va = [h["val_acc"]    for h in hist_d1]
    d2_va = [h["val_acc"]    for h in hist_d2]
    ax.plot(d1_gc, d1_va, "o-", label="D1 (single glance)", markersize=8)
    ax.plot(d2_gc, d2_va, "s-", label="D2 (confidence halting)", markersize=8)
    if hist_d3:
        ax.plot([h["avg_glances"] for h in hist_d3], [h["val_acc"] for h in hist_d3],
                 "^-", label="D3 (REINFORCE, fixed 2 glances)", markersize=8)
    ax.set_xlabel("Avg glances per image"); ax.set_ylabel("Val accuracy")
    ax.set_title(f"Efficiency curve: accuracy vs compute [{dataset_name}]\n(glances = FLOPs proxy)")
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    eff_path = os.path.join(CFG["result_dir"], f"04_efficiency_curve_{dataset_name}.png")
    common.save_figure(fig, eff_path, dpi=130)
    plt.close(fig)
    print(f"Saved: {eff_path}")

    # Figure 2: glance trajectory visualisation on a few real images
    n_show = 4
    fig2, ax2s = plt.subplots(n_show, len(fixations) + 1,
                               figsize=(3*(len(fixations)+1), 3*n_show))
    mu_t  = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
    std_t = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

    test_batch = next(iter(val_ld))
    x_norm_batch, y_batch = test_batch
    x_raw_batch = (x_norm_batch * std_t + mu_t).clamp(0.0, 1.0)

    S = geom["image_size"]

    for row_i in range(min(n_show, x_raw_batch.size(0))):
        img_raw = x_raw_batch[row_i]   # [3, H, W]

        ax2s[row_i][0].imshow(img_raw.permute(1,2,0).numpy().clip(0,1), interpolation="nearest")
        ax2s[row_i][0].set_title("Original", fontsize=8)
        ax2s[row_i][0].axis("off")

        for col_j, fix in enumerate(fixations[:len(fixations)], start=1):
            fov_img = apply_rblur(img_raw, fix, CFG["rblur_sigma0"], geom["rblur_slope"], geom["ppd"])
            ax = ax2s[row_i][col_j]
            ax.imshow(fov_img.permute(1,2,0).numpy().clip(0,1), interpolation="nearest")
            fy_px = int(fix[0] * S); fx_px = int(fix[1] * S)
            ax.plot(fx_px, fy_px, "r+", markersize=10, markeredgewidth=2)
            if row_i == 0:
                ax.set_title(f"Glance {col_j}\nfix=({fix[0]:.2f},{fix[1]:.2f})", fontsize=7)
            ax.axis("off")

    fig2.suptitle(f"Glance trajectories: R-Blur at each fixation location [{dataset_name}]",
                  fontsize=10, y=1.01)
    plt.tight_layout()
    traj_path = os.path.join(CFG["result_dir"], f"04_glance_trajectory_{dataset_name}.png")
    common.save_figure(fig2, traj_path, dpi=130)
    plt.close(fig2)
    print(f"Saved: {traj_path}")

## Write `src/it_feedback.py`

The cell below writes the production module used by NB06 (full model).


In [ ]:
_itfb_content = '''"""
IT feedback and multi-glance loop components.

Public API
----------
FixationGrid           -- regular grid of fixation locations
MultiGlanceFoveatedModel -- D2: differentiable multi-glance (confidence halting)
FixationPolicy         -- D3: learned REINFORCE fixation policy
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from src.foveation import apply_rblur


class FixationGrid:
    """Regular fixation grid, centre fixation always first."""
    def __init__(self, n_rows=2, n_cols=2, margin=0.25):
        self.n_rows = n_rows; self.n_cols = n_cols; self.margin = margin
    def get_fixations(self):
        m = self.margin
        ys = [m + i / max(self.n_rows-1, 1)*(1-2*m) for i in range(self.n_rows)]
        xs = [m + j / max(self.n_cols-1, 1)*(1-2*m) for j in range(self.n_cols)]
        grid = [(y, x) for y in ys for x in xs]
        return sorted(grid, key=lambda p: abs(p[0]-0.5)+abs(p[1]-0.5))


class MultiGlanceFoveatedModel(nn.Module):
    """
    Differentiable multi-glance foveated classifier (D2).
    Backbone receives raw [0,1] pixel tensors.
    """
    def __init__(self, backbone_fn, fixations, halt_threshold=0.75,
                 sigma0=0.5, slope=1.5, ppd=4.0):
        super().__init__()
        self.backbone        = backbone_fn
        self.fixations       = fixations
        self.halt_threshold  = halt_threshold
        self.sigma0 = sigma0; self.slope = slope; self.ppd = ppd
    def foveate(self, x, fixation_yx):
        return torch.stack([apply_rblur(x[i], fixation_yx, self.sigma0,
                                        self.slope, self.ppd) for i in range(x.size(0))])
    def forward(self, x_raw, return_glance_count=False):
        B = x_raw.size(0)
        logit_sum = None; halted = torch.zeros(B, dtype=torch.bool, device=x_raw.device)
        glance_count = torch.ones(B, dtype=torch.long, device=x_raw.device)
        for t, fix in enumerate(self.fixations):
            fov_x = self.foveate(x_raw, fix)
            z_t   = self.backbone(fov_x)
            logit_sum = z_t if logit_sum is None else logit_sum + z_t
            if not self.training and t < len(self.fixations) - 1:
                conf = F.softmax(logit_sum / (t+1), dim=1).max(1).values
                newly = (conf >= self.halt_threshold) & (~halted)
                glance_count += (~halted & ~newly).long()
                halted |= newly
                if halted.all(): break
        final = logit_sum / len(self.fixations)
        return (final, glance_count) if return_glance_count else final


class FixationPolicy(nn.Module):
    """REINFORCE policy: state (logits) -> categorical over fixation locations."""
    def __init__(self, logit_dim, n_fixations):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(logit_dim, 32), nn.ReLU(), nn.Linear(32, n_fixations))
        self.baseline = 0.0
    def forward(self, state):
        return F.softmax(self.net(state), dim=-1)'''
_itfb_path = os.path.join(PROJECT_ROOT, 'src', 'it_feedback.py')
with open(_itfb_path, 'w') as _f: _f.write(_itfb_content)
print(f'Written: {_itfb_path}')

# Verify import
import importlib.util as _ilu
_spec = _ilu.spec_from_file_location('it_feedback', _itfb_path)
_mod  = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
assert hasattr(_mod, 'MultiGlanceFoveatedModel'), 'MultiGlanceFoveatedModel missing'
assert hasattr(_mod, 'FixationGrid'),             'FixationGrid missing'
print('src/it_feedback.py: 3 public symbols verified')

In [ ]:
# ── Save results ─────────────────────────────────────────────────────────
output = {
    "notebook": "04_it_feedback_multiglance",
    "cfg": {k: v for k, v in CFG.items() if not k.endswith("_dir")},
    "D1": {"history_by_dataset": hist_d1_by_dataset,
           "note": ("smoke_test=True: real data if present, else synthetic pipeline check"
                    if CFG["smoke_test"] else "Full run: real data.")},
    "D2": {"history_by_dataset": hist_d2_by_dataset,
           "note": ("smoke_test=True: real data if present, else synthetic pipeline check"
                    if CFG["smoke_test"] else "Full run: real data.")},
    "D3": {"history_by_dataset": hist_d3_by_dataset,
           "note": ("smoke_test=True: real data if present, else synthetic pipeline check"
                    if CFG["smoke_test"] else "Full run: real data. REINFORCE policy on top "
                    "of D1's frozen backbone.")},
    "D4": {"note": "TODO: entropy halting + counterfactual loss (P3)"},
}
rpath = os.path.join(CFG["result_dir"], "04_metrics.json")
common.save_json(output, rpath)
print(f"Results: {rpath}")
for dataset_name in hist_d1_by_dataset:
    print(f"04_efficiency_curve_{dataset_name}.png   : "
          f"{os.path.exists(os.path.join(CFG['result_dir'], f'04_efficiency_curve_{dataset_name}.png'))}")
    print(f"04_glance_trajectory_{dataset_name}.png  : "
          f"{os.path.exists(os.path.join(CFG['result_dir'], f'04_glance_trajectory_{dataset_name}.png'))}")
print(f"src/it_feedback.py        : {os.path.exists(os.path.join(PROJECT_ROOT,'src','it_feedback.py'))}")
print("\n── Notebook 04 complete ✓")